# Lab 5: Generative Modeling

Two ways to generate an image, one lab. You write the three routines they turn on:
the beta-VAE objective with its analytic KL, the forward-diffusion step that noises
an image, and one DDIM step that walks it back. Then you run a controlled VAE study,
design a VAE to a budget, choose one KL-weight setting, and watch a small diffusion
model sample at two step budgets. Everything runs on FashionMNIST.

**What you will do**

1. Write the beta-VAE objective, including its analytic Gaussian KL.
2. Write the forward-diffusion step that sets the denoiser's target.
3. Write one deterministic DDIM reverse step for sampling.
4. Design a VAE to a parameter budget, then change one KL weight.

The red **Your Task** banners mark the parts you complete. Everything else is
provided and ready to run.

## At a glance

| | |
|---|---|
| Stack | PyTorch and torchvision; a GPU runtime is recommended |
| You write | `vae_objective`, `q_sample`, `ddim_step` |
| You submit | Five values, R1 through R5, from the final cell |
| GPU runtime | A few minutes of training in quick mode; longer on the full split |

<div style="border-left:6px solid #A31F34;background:#fff5f6;color:#111827;padding:14px 18px;border-radius:8px;margin:18px 0;">
<div style="color:#A31F34;font-weight:700;text-transform:uppercase;letter-spacing:0.06em;font-size:0.78rem;">Big picture</div>
<p style="margin:6px 0 0;">A VAE and a diffusion model reach the same goal by opposite routes: one compresses an image to a latent and decodes it, the other noises the image and learns to walk it back. Writing the core step of each makes that contrast concrete.</p>
</div>

## What is graded

R1 through R3 check your three functions on fixed inputs. R4 accepts your VAE design
as long as its parameter count lands in the stated range. R5 is a pass-or-fail flag on
your four VAE runs, and because those runs use a held-out split, it also rewards
held-out discipline. Measured losses, runtimes, and sample quality are never submitted;
you read them to interpret your runs, in quick or full mode alike.

<div style="border-left:6px solid #B45309;background:#fffbeb;color:#111827;padding:14px 18px;border-radius:8px;margin:18px 0;">
<div style="color:#B45309;font-weight:700;text-transform:uppercase;letter-spacing:0.06em;font-size:0.78rem;">Watch out</div>
<p style="margin:6px 0 0;">Sharper reconstructions do not always mean a better model. A tiny KL weight can reconstruct well yet leave gaps in the latent space that turn into nonsense when you sample the prior. Judge the tradeoff, not the reconstructions alone.</p>
</div>

In [ ]:
import math
import random
from dataclasses import asdict, dataclass, replace

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

SEED = 7960
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def reset_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


plt.rcParams.update({
    "figure.figsize": (7, 4),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
    "axes.prop_cycle": plt.cycler(
        color=["#A31F34", "#1D4ED8", "#059669", "#B45309", "#7C3AED"]
    ),
})


reset_seed()
print(f"PyTorch {torch.__version__}; device = {DEVICE}")

In [ ]:
QUICK_MODE = True
DATA_ROOT = "./data"

TRAIN_PER_CLASS = 300
VALIDATION_PER_CLASS = 100 if QUICK_MODE else 300
DEFAULT_BATCH_SIZE = 128
EPOCHS = 3 if QUICK_MODE else 8
NUM_WORKERS = 0 if QUICK_MODE else 2

print(f"quick_mode={QUICK_MODE}, epochs={EPOCHS}")

In [ ]:
transform = transforms.ToTensor()
split_source = datasets.FashionMNIST(DATA_ROOT, train=True, download=True)


def stratified_split(targets, train_per_class, validation_per_class, seed):
    target_tensor = torch.as_tensor(targets)
    generator = torch.Generator().manual_seed(seed)
    train_indices, validation_indices = [], []
    for class_id in range(10):
        class_indices = torch.where(target_tensor == class_id)[0]
        order = torch.randperm(len(class_indices), generator=generator)
        class_indices = class_indices[order]
        needed = train_per_class + validation_per_class
        train_indices.extend(class_indices[:train_per_class].tolist())
        validation_indices.extend(class_indices[train_per_class:needed].tolist())
    return sorted(train_indices), sorted(validation_indices)


train_indices, validation_indices = stratified_split(
    split_source.targets, TRAIN_PER_CLASS, VALIDATION_PER_CLASS, SEED
)
assert set(train_indices).isdisjoint(validation_indices)

image_source = datasets.FashionMNIST(
    DATA_ROOT, train=True, transform=transform, download=False
)
train_set = Subset(image_source, train_indices)
validation_set = Subset(image_source, validation_indices)


def seeded_loader(dataset, *, batch_size, shuffle, seed=SEED):
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle, drop_last=False,
        num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda", generator=generator,
    )


def trainable_parameter_count(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


print("training examples:", len(train_indices),
      "validation examples:", len(validation_indices))

<div style="border-left:5px solid #A31F34;padding-left:12px;margin:26px 0 6px;">
<div style="color:#A31F34;font-weight:700;font-size:0.72rem;letter-spacing:0.11em;">YOUR TASK 1</div>
<div style="font-weight:700;font-size:1.2rem;">Write <code>vae_objective</code></div>
</div>

Complete `vae_objective`. Return `(loss, reconstruction, kl)` where the
reconstruction term is the mean over the batch of the per-example squared error
summed over pixels, the KL is the mean over the batch of
`-0.5 * sum(1 + logvar - mu^2 - exp(logvar))`, and `loss = reconstruction + beta * kl`.

In [ ]:
def vae_objective(reconstructed, x, mu, logvar, beta):
    """Return (loss, reconstruction, kl) for the beta-VAE ELBO."""
    # STUDENT TASK 1:
    #   reconstruction = mean over batch of sum over pixels of (reconstructed - x)^2
    #   kl = mean over batch of -0.5 * sum(1 + logvar - mu^2 - exp(logvar))
    #   loss = reconstruction + beta * kl
    raise NotImplementedError("Return (loss, reconstruction, kl).")

In [ ]:
kl_probe_mu = torch.tensor([[0.0, 1.0]])
kl_probe_logvar = torch.tensor([[0.0, math.log(4.0)]])
kl_probe_x = torch.zeros(1, 2)
_, kl_probe_reconstruction, kl_probe_kl = vae_objective(
    kl_probe_x, kl_probe_x, kl_probe_mu, kl_probe_logvar, beta=1.0
)
kl_probe_value = round(kl_probe_kl.item(), 5)
kl_probe_contract = (
    abs(kl_probe_reconstruction.item()) < 1e-9
    and abs(kl_probe_value - 1.30685) < 1e-4
)
assert kl_probe_contract, "KL of mu=(0,1), logvar=(0,ln4) is 0.5*(4 - ln4) = 1.30685."
print("Task 1 checks passed. Analytic KL:", kl_probe_value)

<div style="border-left:5px solid #A31F34;padding-left:12px;margin:26px 0 6px;">
<div style="color:#A31F34;font-weight:700;font-size:0.72rem;letter-spacing:0.11em;">YOUR TASK 2</div>
<div style="font-weight:700;font-size:1.2rem;">Write <code>q_sample</code></div>
</div>

Complete `q_sample`, the forward process
`x_t = sqrt(alpha_bar) * x_0 + sqrt(1 - alpha_bar) * noise`. This defines the noisy
inputs whose noise the denoiser learns to predict.

In [ ]:
def q_sample(x_0, alpha_bar, noise):
    """Forward diffusion: x_t = sqrt(alpha_bar) x_0 + sqrt(1 - alpha_bar) noise."""
    # STUDENT TASK 2: scale the clean signal by sqrt(alpha_bar) and the noise by
    # sqrt(1 - alpha_bar), then add them.
    raise NotImplementedError("Return the noised x_t.")

In [ ]:
qsample_alpha_bar = torch.tensor(0.36)
qsample_x0 = torch.tensor([2.0])
qsample_noise = torch.tensor([0.5])
qsample_x_t = q_sample(qsample_x0, qsample_alpha_bar, qsample_noise)
qsample_probe_value = round(qsample_x_t.item(), 4)
qsample_probe_contract = abs(qsample_probe_value - 1.6) < 1e-4
assert qsample_probe_contract, "sqrt(0.36)*2.0 + sqrt(0.64)*0.5 = 0.6*2 + 0.8*0.5 = 1.6."
print("Task 2 checks passed. Noised value x_t:", qsample_probe_value)

<div style="border-left:5px solid #A31F34;padding-left:12px;margin:26px 0 6px;">
<div style="color:#A31F34;font-weight:700;font-size:0.72rem;letter-spacing:0.11em;">YOUR TASK 3</div>
<div style="font-weight:700;font-size:1.2rem;">Write <code>ddim_step</code></div>
</div>

Complete `ddim_step`. Estimate the clean signal
`x0_hat = (x_t - sqrt(1 - alpha_bar_t) * predicted_noise) / sqrt(alpha_bar_t)`, then
return the less-noisy state
`sqrt(alpha_bar_prev) * x0_hat + sqrt(1 - alpha_bar_prev) * predicted_noise`.

<details style="border:1px solid #e5e7eb;border-radius:8px;padding:10px 14px;background:#f9fafb;color:#111827;margin:14px 0;">
<summary style="cursor:pointer;font-weight:600;">Two steps, in order</summary>
<div style="margin-top:8px;">First recover a clean-image estimate <code>x0_hat</code> from <code>x_t</code> and the predicted noise, then re-noise that estimate to the previous, less-noisy timestep. The probe uses the true noise, so <code>x0_hat</code> should come out exactly 2.0.</div>
</details>

In [ ]:
def ddim_step(x_t, predicted_noise, alpha_bar_t, alpha_bar_prev):
    """One deterministic DDIM reverse step from timestep t to a previous timestep."""
    # STUDENT TASK 3:
    #   x0_hat = (x_t - sqrt(1 - alpha_bar_t) * predicted_noise) / sqrt(alpha_bar_t)
    #   return sqrt(alpha_bar_prev) * x0_hat + sqrt(1 - alpha_bar_prev) * predicted_noise
    raise NotImplementedError("Return the less-noisy x at the previous timestep.")

In [ ]:
ddim_x_t = torch.tensor([1.6])
ddim_noise = torch.tensor([0.5])
ddim_alpha_bar_t = torch.tensor(0.36)
ddim_alpha_bar_prev = torch.tensor(0.64)
ddim_x_prev = ddim_step(ddim_x_t, ddim_noise, ddim_alpha_bar_t, ddim_alpha_bar_prev)
ddim_probe_value = round(ddim_x_prev.item(), 4)
ddim_probe_contract = abs(ddim_probe_value - 1.9) < 1e-4
assert ddim_probe_contract, "With the true noise, x0_hat recovers 2.0 and the step returns 1.9."
print("Task 3 checks passed. DDIM reverse value:", ddim_probe_value)

## Provided: VAE rails

Every VAE run uses your `vae_objective`, the same seed, and a held-out validation
split for held-out ELBO. The encoder/decoder width comes from the design you choose
in task 4.

In [ ]:
class VAE(nn.Module):
    def __init__(self, hidden_dim, latent_dim, input_dim=784):
        super().__init__()
        self.enc_fc1 = nn.Linear(input_dim, hidden_dim)
        self.enc_mu = nn.Linear(hidden_dim, latent_dim)
        self.enc_logvar = nn.Linear(hidden_dim, latent_dim)
        self.dec_fc1 = nn.Linear(latent_dim, hidden_dim)
        self.dec_out = nn.Linear(hidden_dim, input_dim)

    def forward(self, x):
        h = torch.relu(self.enc_fc1(x))
        mu, logvar = self.enc_mu(h), self.enc_logvar(h)
        z = mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)
        reconstructed = self.dec_out(torch.relu(self.dec_fc1(z)))
        return reconstructed, mu, logvar


def build_vae(hidden_dim, latent_dim):
    return VAE(hidden_dim, latent_dim)


@dataclass(frozen=True)
class VAEExperiment:
    hidden_dim: int = 64
    latent_dim: int = 16
    beta: float = 1.0
    learning_rate: float = 1e-3
    batch_size: int = DEFAULT_BATCH_SIZE
    epochs: int = EPOCHS
    seed: int = SEED


def changed_config_fields(reference, candidate):
    reference_fields = asdict(reference)
    candidate_fields = asdict(candidate)
    return {
        name for name in reference_fields
        if reference_fields[name] != candidate_fields[name]
    }


def history_is_complete(record):
    history = record["history"]
    epochs = record["config"]["epochs"]
    return all(
        len(history[metric]) == epochs
        for metric in ("train_loss", "train_kl", "validation_loss", "validation_kl")
    )


@torch.no_grad()
def evaluate_vae(model, loader, beta):
    model.eval()
    total_loss = total_kl = batches = 0.0
    for images, _ in loader:
        x = images.view(images.shape[0], -1).to(DEVICE)
        reconstructed, mu, logvar = model(x)
        loss, _, kl = vae_objective(reconstructed, x, mu, logvar, beta)
        total_loss += loss.item(); total_kl += kl.item(); batches += 1
    return total_loss / batches, total_kl / batches


def run_vae(label, config):
    # Train one VAE configuration with your vae_objective and return its config and full
    # history, including a held-out ELBO and KL each epoch. R5 later checks these histories.
    reset_seed(config.seed)
    model = build_vae(config.hidden_dim, config.latent_dim).to(DEVICE)
    fit_loader = seeded_loader(
        train_set, batch_size=config.batch_size, shuffle=True, seed=config.seed
    )
    train_eval_loader = seeded_loader(
        train_set, batch_size=config.batch_size, shuffle=False, seed=config.seed
    )
    validation_loader = seeded_loader(
        validation_set, batch_size=config.batch_size, shuffle=False, seed=config.seed
    )
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate)
    history = {"train_loss": [], "train_kl": [], "validation_loss": [], "validation_kl": []}
    for epoch in range(1, config.epochs + 1):
        model.train()
        for images, _ in fit_loader:
            x = images.view(images.shape[0], -1).to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            reconstructed, mu, logvar = model(x)
            loss, _, _ = vae_objective(reconstructed, x, mu, logvar, config.beta)
            loss.backward(); optimizer.step()
        train_loss, train_kl = evaluate_vae(model, train_eval_loader, config.beta)
        validation_loss, validation_kl = evaluate_vae(model, validation_loader, config.beta)
        history["train_loss"].append(train_loss)
        history["train_kl"].append(train_kl)
        history["validation_loss"].append(validation_loss)
        history["validation_kl"].append(validation_kl)
        print(f"{label:>22} | epoch {epoch}/{config.epochs} | "
              f"held-out ELBO {validation_loss:.2f} | held-out KL {validation_kl:.2f}")
    record = {"label": label, "config": asdict(config), "history": history}
    del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return record


baseline_config = VAEExperiment()
baseline_result = run_vae("vae baseline", baseline_config)
anchor_config = replace(baseline_config, latent_dim=2)
anchor_result = run_vae("small-latent anchor", anchor_config)

<div style="border-left:5px solid #A31F34;padding-left:12px;margin:26px 0 6px;">
<div style="color:#A31F34;font-weight:700;font-size:0.72rem;letter-spacing:0.11em;">YOUR TASK 4</div>
<div style="font-weight:700;font-size:1.2rem;">Design a VAE</div>
</div>

Choose `hidden_dim` so the VAE has **100,000 to 800,000** trainable parameters. The
VAE has `1570 * hidden_dim + 3 * 16 * hidden_dim + 2 * 16 + 784` parameters at the
default latent dimension of 16.

In [ ]:
# STUDENT TASK 4: pick a positive hidden_dim inside the budget.
hidden_dim = 0

if hidden_dim <= 0:
    raise ValueError("Choose a positive hidden_dim.")
learner_config = replace(baseline_config, hidden_dim=hidden_dim)
learner_vae = build_vae(learner_config.hidden_dim, learner_config.latent_dim)
learner_vae_parameter_count = trainable_parameter_count(learner_vae)
learner_parameter_count = learner_vae_parameter_count
if not (100_000 <= learner_parameter_count <= 800_000):
    raise ValueError("Choose hidden_dim so the VAE has 100,000-800,000 parameters.")
print(f"learner VAE parameters: {learner_parameter_count:,}")

In [ ]:
learner_result = run_vae("learner vae design", learner_config)
for record in (baseline_result, anchor_result, learner_result):
    epochs = np.arange(1, len(record["history"]["validation_loss"]) + 1)
    plt.plot(epochs, record["history"]["validation_loss"], marker="o", label=record["label"])
plt.xlabel("epoch"); plt.ylabel("held-out ELBO loss")
plt.legend(); plt.tight_layout(); plt.show()

<div style="border-left:5px solid #A31F34;padding-left:12px;margin:26px 0 6px;">
<div style="color:#A31F34;font-weight:700;font-size:0.72rem;letter-spacing:0.11em;">YOUR TASK 5</div>
<div style="font-weight:700;font-size:1.2rem;">Choose one KL-weight setting</div>
</div>

Beta sets the reconstruction-regularization tradeoff. Predict what each choice does
to reconstruction sharpness and prior-sample coherence, then choose exactly one beta
change relative to the baseline: `low_beta` (0.25), `high_beta` (4.0), or `tiny_beta`
(0.1). The contract permits exactly one changed field.

In [ ]:
# STUDENT TASK 5: choose "low_beta", "high_beta", or "tiny_beta".
BETA_CHOICE = ""

BETA_OVERRIDES = {
    "low_beta": {"beta": 0.25},
    "high_beta": {"beta": 4.0},
    "tiny_beta": {"beta": 0.1},
}
BETA_LABELS = {
    "low_beta": "chosen beta: 0.25",
    "high_beta": "chosen beta: 4.0",
    "tiny_beta": "chosen beta: 0.1",
}
if BETA_CHOICE not in BETA_OVERRIDES:
    raise ValueError("Choose low_beta, high_beta, or tiny_beta.")
treatment_config = replace(baseline_config, **BETA_OVERRIDES[BETA_CHOICE])
assert changed_config_fields(baseline_config, treatment_config) == {"beta"}

In [ ]:
treatment_result = run_vae(BETA_LABELS[BETA_CHOICE], treatment_config)
for record in (baseline_result, treatment_result):
    epochs = np.arange(1, len(record["history"]["validation_kl"]) + 1)
    plt.plot(epochs, record["history"]["validation_kl"], marker="o", label=record["label"])
plt.xlabel("epoch"); plt.ylabel("held-out KL")
plt.legend(); plt.tight_layout(); plt.show()

## Provided: diffusion arc (ungraded samples)

This arc trains a small denoiser whose target is built by your `q_sample`, then
generates fixed-seed samples with your `ddim_step` at two sampling budgets. The
images are evidence for reflection, not graded values.

In [ ]:
DIFFUSION_STEPS = 50
diffusion_betas = torch.linspace(1e-4, 0.02, DIFFUSION_STEPS)
ALPHA_BAR = torch.cumprod(1.0 - diffusion_betas, dim=0).to(DEVICE)


class Denoiser(nn.Module):
    def __init__(self, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(784 + 1, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 784),
        )

    def forward(self, x_t, t):
        t_feature = (t.float() / DIFFUSION_STEPS).view(-1, 1)
        return self.net(torch.cat([x_t, t_feature], dim=1))


def train_diffusion(seed=SEED, epochs=EPOCHS):
    # Train the denoiser to predict the noise added by your q_sample: draw a timestep and
    # noise, form the noised x_t, and regress the model's output onto that noise.
    reset_seed(seed)
    model = Denoiser().to(DEVICE)
    loader = seeded_loader(train_set, batch_size=DEFAULT_BATCH_SIZE, shuffle=True, seed=seed)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    for epoch in range(1, epochs + 1):
        model.train()
        last = 0.0
        for images, _ in loader:
            x_0 = images.view(images.shape[0], -1).to(DEVICE)
            t = torch.randint(0, DIFFUSION_STEPS, (x_0.shape[0],), device=DEVICE)
            noise = torch.randn_like(x_0)
            alpha_bar = ALPHA_BAR[t].view(-1, 1)
            x_t = q_sample(x_0, alpha_bar, noise)  # your forward step builds the target
            loss = ((model(x_t, t) - noise) ** 2).mean()
            optimizer.zero_grad(set_to_none=True); loss.backward(); optimizer.step()
            last = loss.item()
        print(f"diffusion epoch {epoch}/{epochs} | noise MSE {last:.4f}")
    return model.eval()


@torch.no_grad()
def ddim_sample(model, n_samples, sampling_steps):
    # Generate from pure noise by repeatedly applying your ddim_step along a schedule.
    # Fewer steps is faster but coarser; that is the tradeoff the sampling comparison shows.
    reset_seed(SEED)
    x = torch.randn(n_samples, 784, device=DEVICE)
    schedule = torch.linspace(DIFFUSION_STEPS - 1, 0, sampling_steps).round().long()
    for i in range(len(schedule)):
        t = schedule[i]
        t_prev = schedule[i + 1] if i + 1 < len(schedule) else torch.tensor(0)
        predicted = model(x, t.repeat(n_samples).to(DEVICE))
        x = ddim_step(x, predicted, ALPHA_BAR[t], ALPHA_BAR[t_prev])
    return x


denoiser = train_diffusion()
samples_25 = ddim_sample(denoiser, 8, sampling_steps=25)
samples_100 = ddim_sample(denoiser, 8, sampling_steps=100)
fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for column in range(8):
    axes[0, column].imshow(samples_25[column].view(28, 28).cpu(), cmap="gray")
    axes[1, column].imshow(samples_100[column].view(28, 28).cpu(), cmap="gray")
    axes[0, column].axis("off"); axes[1, column].axis("off")
axes[0, 0].set_ylabel("25 steps"); axes[1, 0].set_ylabel("100 steps")
plt.tight_layout(); plt.show()
print("Fixed-seed samples at 25 vs 100 DDIM steps (ungraded; compare sharpness visually).")

## Pause and reflect (ungraded)

Which beta setting sharpened reconstructions at the cost of prior-sample coherence?
Did 100 DDIM steps look meaningfully better than 25? Name one caveat that keeps each
conclusion tentative and one single-factor experiment you would run next. This is a
place to pause, not a response field.

## Report values

Run the cell below after completing all five tasks and all four VAE runs. R1-R3
verify your three functions on fixed probes. R4 is **your** valid VAE design's
parameter count in the 100,000-800,000 range. R5 verifies the disjoint held-out
split, single-factor comparisons, and complete histories. Enter R1-R5 in the first
Lab 5 submission problem. No measured loss or sample-quality value is submitted.

In [ ]:
run_config_pairs = (
    (baseline_result, baseline_config),
    (anchor_result, anchor_config),
    (learner_result, learner_config),
    (treatment_result, treatment_config),
)

workflow_contract = int(
    set(train_indices).isdisjoint(validation_indices)
    and len({record["label"] for record, _ in run_config_pairs}) == 4
    and all(record["config"] == asdict(config) for record, config in run_config_pairs)
    and all(history_is_complete(record) for record, _ in run_config_pairs)
    and changed_config_fields(baseline_config, anchor_config) == {"latent_dim"}
    and changed_config_fields(baseline_config, learner_config) == {"hidden_dim"}
    and changed_config_fields(baseline_config, treatment_config) == {"beta"}
    and learner_parameter_count == learner_vae_parameter_count
    and 100_000 <= learner_parameter_count <= 800_000
)

probe_kl = round(kl_probe_value, 5) if kl_probe_contract else -1.0
probe_forward = round(qsample_probe_value, 4) if qsample_probe_contract else -1.0
probe_reverse = round(ddim_probe_value, 4) if ddim_probe_contract else -1.0

report_values = {
    "R1: analytic VAE KL on the fixed probe": probe_kl,
    "R2: forward-diffusion noised value on the fixed probe": probe_forward,
    "R3: DDIM reverse-step value on the fixed probe": probe_reverse,
    "R4: learner-designed VAE parameter count": learner_parameter_count,
    "R5: experimental workflow contract": workflow_contract,
}
assert abs(probe_kl - 1.30685) < 1e-4
assert abs(probe_forward - 1.6) < 1e-4
assert abs(probe_reverse - 1.9) < 1e-4
assert 100_000 <= learner_parameter_count <= 800_000
assert workflow_contract == 1

print("LAB 5 REPORT VALUES")
for label, value in report_values.items():
    print(f"{label}: {value}")